# power_brick — Dual-Rail 12V → 5V / 3.3V Power Supply Brick

12V barrel/screw input → Buck #1 (TPS54302, 5V@2A) + Buck #2 (TPS54302, 3.3V@1A). Both rails exposed via 2-pin output headers.

In [ ]:
import hw_toolkit as hw
from hw_toolkit.parts import Buck
import pathlib

board = hw.Board("power_brick")
board

In [ ]:
# 12V input connector (2-pin screw/barrel terminal)
# Connector_Generic:Conn_01x02 resolves to a real KiCad symbol.
# Pin_1 = VIN (12V), Pin_2 = GND
j_in = board.module(
    id="j_in",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_in

In [ ]:
# Buck #1: 12V -> 5V @ 2A
# TPS54302 Vref = 0.8V: Vout = Vref * (1 + Rtop/Rbot)
# For 5V: Rtop/Rbot = (5.0/0.8) - 1 = 5.25
# Rtop=52.3k, Rbot=10k => Vout = 0.8 * (1 + 52.3/10) = 0.8 * 6.23 = 4.984V ≈ 5V
buck_5v = Buck(
    board,
    id="buck_5v",
    mpn="TPS54302",
    package="SOT-23-6",
    vin=12.0,
    vout=5.0,
    l="10uH",
    cin="10uF",
    cout="22uF",
    cboot="100nF",
    rtop="52.3k",
    rbot="10k",
    cap_package="0805",
    res_package="0603",
    ind_package="1210",
)
buck_5v

In [ ]:
# Buck #2: 12V -> 3.3V @ 1A
# For 3.3V: Rtop/Rbot = (3.3/0.8) - 1 = 3.125
# Rtop=31.6k, Rbot=10k => Vout = 0.8 * (1 + 31.6/10) = 0.8 * 4.16 = 3.328V ≈ 3.3V
buck_3v3 = Buck(
    board,
    id="buck_3v3",
    mpn="TPS54302",
    package="SOT-23-6",
    vin=12.0,
    vout=3.3,
    l="10uH",
    cin="10uF",
    cout="22uF",
    cboot="100nF",
    rtop="31.6k",
    rbot="10k",
    cap_package="0805",
    res_package="0603",
    ind_package="1210",
)
buck_3v3

In [ ]:
# 5V output header
j_5v = board.module(
    id="j_5v",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_5v

In [ ]:
# 3.3V output header
j_3v3 = board.module(
    id="j_3v3",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_3v3

In [ ]:
# --- Net wiring ---
#
# 12V input rail:
# Buck factory creates per-buck VIN nets (buck_5v_vin, buck_3v3_vin).
# We stitch both onto the single 12V node by extending buck_5v_vin
# with the input connector and all of buck_3v3's VIN-side pins.
# (KiCad netlist reconciliation handles the pin appearing in both nets
# by ultimately treating them as one net.)
board.nets["buck_5v_vin"] += (
    "j_in.Pin_1",         # connector 12V positive
    "buck_3v3.VIN",       # 3.3V buck IC VIN pin
    "buck_3v3_cin.1",     # 3.3V buck input cap positive
    "buck_3v3.EN",        # 3.3V buck EN (always-on tie)
)

# GND rail: shared ground net (created by first Buck call)
gnd = board.nets["gnd"]
gnd += "j_in.Pin_2", "j_5v.Pin_2", "j_3v3.Pin_2"

# 5V output: Buck factory created buck_5v_vout with the inductor output
board.nets["buck_5v_vout"] += "j_5v.Pin_1"

# 3.3V output: Buck factory created buck_3v3_vout
board.nets["buck_3v3_vout"] += "j_3v3.Pin_1"

print("12V rail:", board.nets["buck_5v_vin"])
print("GND rail:", board.nets["gnd"])
print("5V rail:", board.nets["buck_5v_vout"])
print("3.3V rail:", board.nets["buck_3v3_vout"])

In [ ]:
board.summary()

In [ ]:
# ERC — all parts (TPS54302, Device:R/C/L, Connector_Generic:Conn_01x02)
# resolve to real KiCad symbols, so gate on ERC_REAL_SYMBOL_CODES (tighter set).
board.check_erc(expected_codes=hw.ERC_REAL_SYMBOL_CODES)

In [ ]:
# Export — absolute path per AGENT_GUIDE §7
out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/power_brick/power_brick.zip")
board.export_kicad(out, unzip=True, expected_codes=hw.ERC_REAL_SYMBOL_CODES)
print("Exported:", out)
assert out.exists(), "zip not created!"
print("ZIP size:", out.stat().st_size, "bytes")